# Export-Pipeline Smoke Test (no training)

Verifies the whole post-training pipeline on current Colab — install,
checkpoint load, ONNX export, OpenVINO conversion, NNCF INT8, pickle —
using the **pretrained COCO RTMDet-tiny** checkpoint instead of a
training run. GPU runtime recommended, finishes in a few minutes.

Note: calibration images here are synthetic noise, so the stability
check will usually say "Optimization unstable — packaging standard
version only". That is EXPECTED in this smoke test — the point is that
every stage runs without crashing.

## 1. Install (identical to the training notebooks)

In [ ]:
# ── Current Colab (Python 3.13+): stock torch/numpy, NOTHING downgraded ─────
# mmcv is the only compiled piece. Dorna hosts a prebuilt wheel in this repo
# (colab_wheels/, built by training_notebooks/build_mmcv_wheel.ipynb).
# When Colab bumps its runtime, rebuild the wheel there and replace it.
import importlib, importlib.util, os, sys, urllib.request

PYTAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEEL_URL = ("https://github.com/dorna-robotics/dorna_vision/raw/pro/colab_wheels/"
             f"mmcv-2.2.0-{PYTAG}-{PYTAG}-linux_x86_64.whl")

def _env_ok(loud=True):
    try:
        import torch, mmcv, mmengine, mmdet
        import openvino, nncf
        from mmcv.ops import MultiScaleDeformableAttention   # the compiled part
        if loud:
            print("Environment OK:")
            print(f"  torch:    {torch.__version__}")
            print(f"  mmcv:     {mmcv.__version__}")
            print(f"  mmdet:    {mmdet.__version__}")
            print(f"  openvino: {openvino.__version__}")
        return True
    except Exception:
        return False

if not _env_ok(loud=True):
    print("Installing (~3-4 min)...")
    # modern build tooling — old setuptools breaks on Python 3.13
    !pip -q install "setuptools>=79,<80" "wheel>=0.45"
    # mmengine via plain pip — NEVER via mim (mim drags in openxlab, which
    # downgrades setuptools/requests/rich and bricks Python 3.13)
    !pip -q install mmengine

    # Prebuilt mmcv matched to this runtime — seconds, no compile
    try:
        urllib.request.urlopen(urllib.request.Request(WHEEL_URL, method="HEAD"), timeout=15)
    except Exception:
        raise SystemExit(
            f"No prebuilt mmcv wheel for this runtime ({PYTAG}) at:\n  {WHEEL_URL}\n"
            "Colab's runtime changed — run training_notebooks/build_mmcv_wheel.ipynb "
            "once and publish the new wheel to colab_wheels/.")
    !pip -q install {WHEEL_URL}

    !pip -q install mmdet==3.3.0

    # Lift the mm* hardcoded version caps (path-resolved, no python3.N paths)
    def _lift_caps(mod, subs):
        spec = importlib.util.find_spec(mod)
        if not spec:
            return
        p = os.path.join(os.path.dirname(spec.origin), "__init__.py")
        t = open(p).read()
        for a, b in subs:
            t = t.replace(a, b)
        open(p, "w").write(t)
    _lift_caps("mmdet", [("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")])

    # OpenVINO + NNCF + ONNX export
    !pip -q install openvino nncf onnx onnxruntime

    importlib.invalidate_caches()
    assert _env_ok(loud=False), "install did not verify — read the log above; do NOT continue"
    print("\n>>> Runtime -> Restart session, then run all cells from the top <<<")


## 2. Stand-in setup (replaces dataset + training)

In [ ]:
import os, glob, json, pickle, urllib.request
import numpy as np, cv2, torch, mmdet
from google.colab import files

# ---- stand-ins for the training cells ----
MODEL_SIZE = "tiny"
IMAGE_SIZE = 640
OPTIMIZE   = True
project_name = "export_smoke"

class _P:            # train notebook's Roboflow project object (colors only)
    colors = {}
project = _P()

# COCO-pretrained RTMDet-tiny as the "trained" checkpoint
work_dir = "/content/work_dirs/export_smoke"
os.makedirs(work_dir, exist_ok=True)
CKPT_URL = ("https://download.openmmlab.com/mmdetection/v3.0/rtmdet/"
            "rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth")
ckpt_path = f"{work_dir}/best_smoke.pth"
if not os.path.exists(ckpt_path):
    urllib.request.urlretrieve(CKPT_URL, ckpt_path)
print("checkpoint:", ckpt_path, f"({os.path.getsize(ckpt_path)/1e6:.1f} MB)")

# the matching config ships inside the mmdet pip package
config_path = os.path.join(os.path.dirname(mmdet.__file__), ".mim",
                           "configs", "rtmdet", "rtmdet_tiny_8xb32-300e_coco.py")
assert os.path.isfile(config_path), config_path
classes = [f"coco_{i}" for i in range(80)]   # only the COUNT matters here

# synthetic calibration images (noise) — enough to exercise NNCF
DATA_ROOT = "/content/smoke_data"
os.makedirs(f"{DATA_ROOT}/train", exist_ok=True)
rng = np.random.default_rng(0)
for i in range(8):
    img = rng.integers(0, 255, (480, 640, 3), dtype=np.uint8)
    cv2.imwrite(f"{DATA_ROOT}/train/calib_{i}.jpg", img)
print("setup done")

## 3. Export & Optimize (verbatim from train_object_detection)

In [ ]:
import cv2
import openvino as ov
import nncf
from mmdet.apis import init_detector

# Pin the LEGACY onnx exporter: torch >= 2.6 defaults dynamo=True, and the
# export wrapper below is written against the legacy tracer.
import torch.onnx
if not hasattr(torch.onnx, "_dynamo_patched"):
    _orig_export = torch.onnx.export
    def _patched_export(*args, **kwargs):
        kwargs["dynamo"] = False
        try:
            return _orig_export(*args, **kwargs)
        except TypeError:      # older torch: no dynamo kwarg
            kwargs.pop("dynamo", None)
            return _orig_export(*args, **kwargs)
    torch.onnx.export = _patched_export
    torch.onnx._dynamo_patched = True

# torch >= 2.6 defaults torch.load(weights_only=True); our OWN checkpoints
# carry numpy scalars in their metadata and are trusted — load them fully.
if not hasattr(torch.serialization, "_wo_patched"):
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
    torch.serialization._wo_patched = True


# ---------- pick best checkpoint ----------
ckpts = sorted(glob.glob(os.path.join(work_dir, "best_*.pth")))
if not ckpts:
    ckpts = sorted(glob.glob(os.path.join(work_dir, "epoch_*.pth")))
assert ckpts, "No checkpoint found!"
checkpoint = ckpts[-1]
print(f"Checkpoint: {checkpoint}")

# Use EMA weights for export (better eval accuracy)
raw_ckpt = torch.load(checkpoint, map_location="cpu")
if "ema_state_dict" in raw_ckpt:
    raw_ckpt["state_dict"] = raw_ckpt["ema_state_dict"]
    print("Using averaged weights")
fixed_ckpt = os.path.join(work_dir, "best_for_export.pth")
torch.save(raw_ckpt, fixed_ckpt)

# ---------- Load model ----------
detector = init_detector(config_path, fixed_ckpt, device="cpu")
detector.eval()


# ---------- Export wrapper: unified YOLO-style output tensor ----------
# Output shape: (batch, total_anchors, 4 + 1 + num_classes)
#               [x_center, y_center, w, h, obj_score, cls_1, cls_2, ...]
# Two alignments with training config:
#  1. mmdet's RTMDet head already multiplies regression by stride — don't double it.
#  2. anchor_generator config uses offset=0 → priors at col*stride (not (col+0.5)*stride).

PRIOR_OFFSET = 0.0   # MUST match MlvlPointGenerator's `offset` in the training config


class DetectorExportWrapper(torch.nn.Module):
    def __init__(self, detector, num_classes, strides=(8, 16, 32), prior_offset=PRIOR_OFFSET):
        super().__init__()
        self.detector = detector
        self.num_classes = num_classes
        self.strides = strides
        self.prior_offset = prior_offset

    def forward(self, x):
        feats = self.detector.extract_feat(x)
        cls_scores, bbox_preds = self.detector.bbox_head(feats)
        outputs = []
        for cls_s, bbox_p, stride in zip(cls_scores, bbox_preds, self.strides):
            B, _, H, W = cls_s.shape
            device = cls_s.device
            yv, xv = torch.meshgrid(
                torch.arange(H, device=device, dtype=torch.float32),
                torch.arange(W, device=device, dtype=torch.float32),
                indexing='ij',
            )
            grid = torch.stack((xv, yv), dim=-1)
            grid = (grid + self.prior_offset) * stride
            grid = grid.view(1, H * W, 2)
            bbox_p = bbox_p.permute(0, 2, 3, 1).reshape(B, H * W, 4)
            x1 = grid[..., 0:1] - bbox_p[..., 0:1]
            y1 = grid[..., 1:2] - bbox_p[..., 1:2]
            x2 = grid[..., 0:1] + bbox_p[..., 2:3]
            y2 = grid[..., 1:2] + bbox_p[..., 3:4]
            xc = (x1 + x2) * 0.5
            yc = (y1 + y2) * 0.5
            w = x2 - x1
            h = y2 - y1
            boxes_xywh = torch.cat([xc, yc, w, h], dim=-1)
            cls_probs = cls_s.permute(0, 2, 3, 1).reshape(B, H * W, self.num_classes).sigmoid()
            obj = cls_probs.max(dim=-1, keepdim=True).values
            outputs.append(torch.cat([boxes_xywh, obj, cls_probs], dim=-1))
        return torch.cat(outputs, dim=1)


wrapped = DetectorExportWrapper(detector, num_classes=len(classes))
wrapped.eval()

os.makedirs("/content/export", exist_ok=True)
onnx_path = "/content/export/model.onnx"
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)

torch.onnx.export(
    wrapped, dummy, onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
)
print(f"Exported model: {onnx_path} ({os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB)")

# ---------- Standard-precision export ----------
core = ov.Core()
ov_model = core.read_model(onnx_path)
std_xml = "/content/export/model.xml"
ov.save_model(ov_model, std_xml)
std_bin = std_xml.replace(".xml", ".bin")
print(f"Standard model: {std_bin} ({os.path.getsize(std_bin) / 1024 / 1024:.1f} MB)")

# ---------- Optimized model (if requested) ----------
MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32)
STD  = np.array([57.375, 57.12, 58.395], dtype=np.float32)

def preprocess_for_calibration(img_path, size):
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(h * r), int(w * r)
    img = cv2.resize(img, (nw, nh))
    padded = np.full((size, size, 3), 128, dtype=np.uint8)
    padded[:nh, :nw] = img
    tensor = ((padded.astype(np.float32) - MEAN) / STD).transpose(2, 0, 1)[None]
    return tensor


opt_xml = None
opt_bin = None

if OPTIMIZE:
    print("\nOptimizing model...")
    calib_imgs = sorted(glob.glob(f"{DATA_ROOT}/train/*.jpg"))[:200]
    calib_tensors = [t for t in (preprocess_for_calibration(p, IMAGE_SIZE) for p in calib_imgs) if t is not None]
    print(f"  Calibration samples: {len(calib_tensors)}")

    ov_model_for_opt = core.read_model(onnx_path)
    calib_dataset = nncf.Dataset(calib_tensors, lambda x: x)
    optimized = nncf.quantize(
        ov_model_for_opt,
        calib_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(calib_tensors),
    )
    opt_xml_candidate = "/content/export/model_opt.xml"
    ov.save_model(optimized, opt_xml_candidate)
    opt_bin_candidate = opt_xml_candidate.replace(".xml", ".bin")
    print(f"  Optimized model: {opt_bin_candidate} ({os.path.getsize(opt_bin_candidate) / 1024 / 1024:.1f} MB)")
    print(f"  Size reduction: {100 * (1 - os.path.getsize(opt_bin_candidate) / os.path.getsize(std_bin)):.0f}%")

    # ---- Detection-specific stability check ----
    # The old "max-value" check fails for detection output because bbox coords
    # (pixel values, ~640) dominate the max, so zero-input and real-input both
    # yield similar max values → false "unstable" verdict.
    # Instead, check *classification confidence* which SHOULD differ between zeros
    # (no objects → low conf) and a real image (objects → high conf).
    compiled_opt = core.compile_model(optimized, "CPU")
    compiled_std = core.compile_model(ov_model, "CPU")
    probe = calib_tensors[0]
    zeros = np.zeros_like(probe)

    def max_detection_score(out):
        # out shape: (1, N, 4+1+C) — peak score = obj * max_cls_prob
        arr = np.asarray(out)
        return float((arr[..., 4] * arr[..., 5:].max(axis=-1)).max())

    o_real_opt = list(compiled_opt([probe]).values())[0]
    o_zero_opt = list(compiled_opt([zeros]).values())[0]
    o_real_std = list(compiled_std([probe]).values())[0]

    real_opt = max_detection_score(o_real_opt)
    zero_opt = max_detection_score(o_zero_opt)
    real_std = max_detection_score(o_real_std)

    response = real_opt - zero_opt
    drift = abs(real_opt - real_std) / max(real_std, 1e-8)
    print(f"  Stability check: real_conf={real_opt:.3f}, zero_conf={zero_opt:.3f}, drift={drift*100:.1f}%")

    # Keep optimized version if:
    #   - Real images produce non-trivial peak confidence (>0.1)
    #   - Optimized peak confidence tracks standard within 50%
    if real_opt > 0.1 and drift < 0.50:
        opt_xml = opt_xml_candidate
        opt_bin = opt_bin_candidate
        print("  Optimization OK — packaging optimized version")
    else:
        print("  Optimization unstable — packaging standard version only")
else:
    print(f"\nOPTIMIZE={OPTIMIZE} — packaging standard version only.")

## 4. Pickle (verbatim from train_object_detection)

In [ ]:
# Pickle format is compatible with the existing detection inference pipeline:
#   { bin, xml, cls, colors, meta }
# If optimization was requested and succeeded, bin/xml contain the optimized model.
# Otherwise they contain the standard model. meta.precision tags which one.

if opt_xml and opt_bin and os.path.exists(opt_xml):
    final_xml_path = opt_xml
    final_bin_path = opt_bin
    precision_tag = "int8"
    variant = "optimized"
else:
    final_xml_path = std_xml
    final_bin_path = std_bin
    precision_tag = "fp32"
    variant = "standard"

with open(final_xml_path, "r", encoding="utf-8") as f:
    xml_data = f.read()
with open(final_bin_path, "rb") as f:
    bin_data = f.read()

model_dict = {
    "bin": bin_data,
    "xml": xml_data,
    "cls": classes,
    "colors": project.colors,
    "meta": {
        "model_type": MODEL_SIZE,
        "type": "od",
        "image_size": IMAGE_SIZE,
        "precision": precision_tag,   # "int8" or "fp32"
    },
}

pickle_path = f"/content/{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Pickle contains {variant} model ({len(bin_data) / 1024 / 1024:.1f} MB)")
print(f"\nSaved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path) / 1024 / 1024:.1f} MB")
print(f"Model size: {MODEL_SIZE}")
print(f"Classes: {classes}")
print(f"Image size: {IMAGE_SIZE}")

files.download(pickle_path)